###Ingestión del archivo "movie.csv"

In [0]:
%run '../includes/configuration'

In [0]:
%run '../includes/common_functions'

In [0]:
dbutils.widgets.text("p_environment","")
v_environment = dbutils.widgets.get("p_environment")

In [0]:
dbutils.widgets.text("p_file_date","2024-12-30")
v_file_date = dbutils.widgets.get("p_file_date")

##### Paso 1 - Leer el archivo usando dataframereader


In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType

In [0]:
movie_schema = StructType(fields=[ 
  StructField("movieId", IntegerType(), False),
  StructField("title", StringType(), True),
  StructField("budget", StringType(), True),
  StructField("homePage", StringType(), True),
  StructField("overview", StringType(), True),
  StructField("popularity", DoubleType(), True),
  StructField("yearReleaseDate", IntegerType(), True),
  StructField("releaseDate", DateType(), True),
  StructField("revenue", DoubleType(), True),
  StructField("durationTime", IntegerType(), True),
  StructField("movieStatus", StringType(), True),
  StructField("tagLine", StringType(), True),
  StructField("voteAverage", DoubleType(), True),
  StructField("voteCount", StringType(), True)
  ])

In [0]:
movie_df = spark.read \
    .option("header",True) \
    .schema(movie_schema) \
    .csv(f"{bronze_folder_path}/{v_file_date}/movie.csv")

In [0]:
from pyspark.sql.functions import col

In [0]:
movies_selected_df = movie_df.select( \
    col("movieId"), \
    col("title"), \
    col("budget"), \
    col("popularity"), \
    col("yearReleaseDate"), \
    col("releaseDate"), \
    col("revenue"), \
    col("durationTime"), \
    col("voteAverage"), \
    col("voteCount") \
)

In [0]:
movies_renamed_df = movies_selected_df \
                    .withColumnRenamed("movieId","movie_id") \
                    .withColumnRenamed("yearReleaseDate","year_release_date") \
                    .withColumnRenamed("releaseDate","release_date") \
                    .withColumnRenamed("durationTime","duration_time") \
                    .withColumnRenamed("voteAverage","vote_average") \
                    .withColumnRenamed("voteCount","vote_count")


In [0]:
from pyspark.sql.functions import current_timestamp, lit

In [0]:
movies_final_df = movies_renamed_df \
                    .withColumn("ingestion_date", current_timestamp()) \
                    .withColumn("environment", lit(v_environment)) \
                    .withColumn("file_date", lit(v_file_date))

In [0]:
#overwrite_partition("movie_silver","movies","file_date",v_file_date)

In [0]:
merge_condition = "tgt.movie_id = src.movie_id AND tgt.file_date = src.file_date"
merge_delta_lake(movies_final_df,"movie_silver","movies",merge_condition,"file_date")

In [0]:
%sql
SELECT file_date, COUNT(1) FROM movie_silver.movies
GROUP BY file_date

file_date,count(1)
2024-12-16,3000
2024-12-23,1000
2024-12-30,725


In [0]:
%sql
DESC EXTENDED movie_silver.movies

col_name,data_type,comment
movie_id,int,null
title,string,null
budget,string,null
popularity,double,null
year_release_date,int,null
release_date,date,null
revenue,double,null
duration_time,int,null
vote_average,double,null
vote_count,string,null
